In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [4]:
cities = ["Berlin", "Hamburg", "Munich"]
cities_data = []

# Headers to make the request look like it comes from a regular web browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

for city in cities:
    url = f"https://en.wikipedia.org/wiki/{city}"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Get the main page title
        title = soup.find(id="firstHeading").text
        
        cities_data.append({
            "city_name": city,
            "title": title,
            "url": url
        })
        print(f"Successfully fetched data for: {city}")
    else:
        print(f"Failed to fetch data for: {city} (Status code: {response.status_code})")

# Convert the list to a Pandas DataFrame
df_cities = pd.DataFrame(cities_data)
print(df_cities)

Successfully fetched data for: Berlin
Successfully fetched data for: Hamburg
Successfully fetched data for: Munich
  city_name    title                                    url
0    Berlin   Berlin   https://en.wikipedia.org/wiki/Berlin
1   Hamburg  Hamburg  https://en.wikipedia.org/wiki/Hamburg
2    Munich   Munich   https://en.wikipedia.org/wiki/Munich


In [7]:
def get_cities_data(city_list):
    # Set headers to simulate a real browser request and avoid 403 HTTP errors
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }
    
    # Initialize an empty list to store dictionary data for each city
    cities_data = []

    # Loop through each city provided in the input list
    for city in city_list:
        # Construct the target Wikipedia URL
        url = f"https://en.wikipedia.org/wiki/{city}"
        
        # Send HTTP GET request to Wikipedia
        response = requests.get(url, headers=headers)
        
        # Check if the page request was successful (HTTP status 200)
        if response.status_code == 200:
            # Parse the HTML content using BeautifulSoup
            soup = BeautifulSoup(response.content, "html.parser")
            
            # Set country statically since all current cities are in Germany
            country = "Germany"
            
            # Find latitude and longitude elements in the parsed HTML
            lat_elem = soup.find("span", {"class": "latitude"})
            lon_elem = soup.find("span", {"class": "longitude"})
            
            # Extract text if element exists, otherwise assign None
            latitude = lat_elem.text if lat_elem else None
            longitude = lon_elem.text if lon_elem else None
            
            # Append collected data as a dictionary to the list
            cities_data.append({
                "city_name": city,
                "country": country,
                "latitude": latitude,
                "longitude": longitude
            })
            print(f"Data fetched successfully for: {city}")
        else:
            print(f"Failed to fetch data for: {city}")
            
    # Convert list of dictionaries into a Pandas DataFrame
    return pd.DataFrame(cities_data)

# Define the target city list
cities = ["Berlin", "Hamburg", "Munich"]

# Execute the function to fetch data
df_cities = get_cities_data(cities)

# Display the resulting DataFrame
print(df_cities)

Data fetched successfully for: Berlin
Data fetched successfully for: Hamburg
Data fetched successfully for: Munich
  city_name  country    latitude   longitude
0    Berlin  Germany  52°31′12″N  13°24′18″E
1   Hamburg  Germany     53°33′N     10°00′E
2    Munich  Germany  48°08′15″N  11°34′30″E


After pulling the coordinates for our cities, we move on the SQL and start creating the Database

In [11]:
pip install sqlalchemy pymysql


Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
# PURPOSE: Connect to MySQL database 'Gans' and insert the cities DataFrame into the 'cities' table.

import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Load environment variables from .env file
load_dotenv()

# Retrieve credentials securely from environment variables
user = "root"
password = os.getenv("MYSQL_PASSWORD")
host = "localhost"
port = "3306"
database = "Gans"

# Create database connection string
connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

# Insert DataFrame records into MySQL table 'cities'
# if_exists='append' appends new records to the existing table structure
df_cities.to_sql(name="cities", con=engine, if_exists="append", index=False)

print("Data successfully inserted into Gans database!")

Data successfully inserted into Gans database!
